## **In Class Activity: Feature Engineering and Regression**

In this notebook, we focus on **feature engineering**, the process of transforming raw variables into more useful inputs for machine learning models. We will use a synthetic dataset so you can clearly see how engineered features change model performance.

## **What is Feature Engineering?**

**Feature engineering** is the practice of creating, transforming, encoding, or selecting features so a model can better capture useful patterns in the data.

Common feature engineering tasks include:

- Exploring data types and distributions
- Encoding categorical variables
- Scaling numeric features
- Extracting components from datetime values
- Reducing redundancy from highly correlated features

## **Why It Matters**

Many models (including linear regression) depend heavily on the quality and representation of the input features. Better features can improve:

- Predictive performance
- Model stability
- Training efficiency
- Interpretability

## **Objectives**

In this activity, you will:
1. Explore a mixed-type dataset (numeric, categorical, and temporal features)
2. Train a baseline linear regression model
3. Apply several feature engineering techniques
4. Compare model performance before and after feature engineering


## **PAIR ASSIGNMENT**

Please work in pairs for this exercise. Pair up with the person next to you. If you find that there isn't anyone sitting next to you or if you're unable to form a pair, please raise your hand, and I will assist in pairing you with someone.


### **Task 0: Please assign your full name and your partner's full name to the variables below, respectively.**


Example:

```
your_name = 'Taylor Swift'
your_partner_name = 'Travis Kelce'
```


In [ ]:
# Assign your and your partner's names here.  If you find that there isn't anyone sitting next to you or if you're unable to form a pair, please raise your hand.
# If, in the end, we are unable to find a partner for you, please assign the word "self" to the `your_partner_name` variable.
your_name = 'Jonathan Cruz'
your_partner_name = ''

### **Dataset**

For the following tasks, we will use a synthetic dataset generated with NumPy in the next cell.

### **Why a Synthetic Dataset?**

A synthetic dataset lets us control the relationships between features and the target, which makes it easier to understand *why* certain feature engineering steps help.

This dataset intentionally includes:
- Multiple numerical features (including correlated features)
- A categorical feature that affects the target
- A temporal feature (`Temporal`) for datetime extraction practice
- A target variable built from a mix of linear, nonlinear, and categorical effects


### **TASK 1: Review the implementation below with your study partner, making sure both of you comprehend its functionality and the underlying mechanics.**

As you review the data generation code, identify which features are:
- directly informative for the target,
- redundant (highly correlated with another feature), and
- non-numeric features that will need preprocessing before modeling.

This will help you anticipate the feature engineering steps used later in the notebook.


In [1]:
# Creating the synthetic dataset
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

# Numerical Features
x1 = np.linspace(-10, 10, n)  # Linear relationship with the target
x2 = x1**2 + np.random.randn(n) * 5  # Quadratic relationship with the target
x3 = np.random.randn(n) * 5 + 25  # Random data
x4 = x1 * 3 + np.random.randn(n) * 2  # Derived from x1, highly correlated

# Categorical Feature
categories = ['A', 'B', 'C', 'D']
weights = [0.1, 0.2, 0.3, 0.4]  # Some categories are more common
categorical = np.random.choice(categories, n, p=weights)

# Temporal Feature
date_range = pd.date_range(start='2021-01-01', periods=n, freq='D')
temporal = np.random.choice(date_range, n)

# Correctly constructing the target variable by handling the categorical values
target_effect_from_category = np.array([categories.index(cat) * 10 for cat in categorical])
target = x1 * 2 + x2 + x3 - target_effect_from_category + np.random.randn(n) * 5

df_complex = pd.DataFrame({
    'x1': x1,
    'x2': x2,
    'x3': x3,
    'x4': x4,
    'Categorical': categorical,
    'Temporal': temporal,
    'Target': target
})

df_complex.head()

,x1,x2,x3,x4,Categorical,Temporal,Target
0,-10.00000,102.483571,31.996777,-31.350357,D,2023-03-14,89.966849
1,-9.97998,98.908679,29.623168,-30.228977,D,2021-04-24,70.743647
2,-9.95996,102.439245,25.298152,-31.464720,B,2023-06-12,82.779315
3,-9.93994,106.417555,21.765316,-30.435743,D,2023-06-16,81.158889
4,-9.91992,97.234044,28.491117,-33.546989,C,2023-03-31,87.103607


### **TASK 2: Explore the Dataset**

Perform an initial exploratory data analysis (EDA) to understand the nature and types of features present. Begin by checking data types, summary statistics, and the number of unique values for each column.

### **What to look for during EDA**

- Which columns are numerical vs categorical vs datetime
- Typical ranges and spread of numeric variables
- Whether any variables appear redundant or derived from others
- The cardinality (number of unique values) of each feature

This quick inspection helps you choose the correct preprocessing steps in later tasks.


In [7]:
# Write your code here
df = df_complex.copy()
print(df.head())

print(df.dtypes)

print(df.describe().T)

for col in df.columns:
    nuni = df[col].nunique(dropna=False)
    sample = df[col].dropna().unique()[:6]
    print(f" - {col}: {nuni} unique; sample -> {sample}")

         x1          x2         x3         x4 Categorical   Temporal  \
0 -10.00000  102.483571  31.996777 -31.350357           D 2023-03-14   
1  -9.97998   98.908679  29.623168 -30.228977           D 2021-04-24   
2  -9.95996  102.439245  25.298152 -31.464720           B 2023-06-12   
3  -9.93994  106.417555  21.765316 -30.435743           D 2023-06-16   
4  -9.91992   97.234044  28.491117 -33.546989           C 2023-03-31   

      Target  
0  89.966849  
1  70.743647  
2  82.779315  
3  81.158889  
4  87.103607  
x1                    float64
x2                    float64
x3                    float64
x4                    float64
Categorical            object
Temporal       datetime64[ns]
Target                float64
dtype: object
           count                        mean                  min  \
x1        1000.0                         0.0                -10.0   
x2        1000.0                   33.496727           -11.564536   
x3        1000.0                   25.354181  

### **TASK 3: Linear Regression Model Before Feature Engineering**

Apply a simple linear regression model on the dataset (without any feature engineering) and compute its performance using the `R^2` score. This will simply be a method from scikit learn (see the import statement) that you will apply to the test and prediction datasets. Name this variable `r2_before`.

This baseline model gives us a reference point so we can measure whether feature engineering improves performance.

* The `R^2` score, also known as the coefficient of determination, is a statistical measure that represents the proportion of the variance in the dependent variable that is predictable from the independent variables. It provides a measure of how well the observed outcomes are replicated by the model based on the provided explanatory variables.

### **Important setup choice**

At this stage, intentionally exclude the categorical and temporal columns so the baseline model only uses the raw numerical features.


In [ ]:
# Write your code here
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

X_before = df[['x1','x2','x3','x4']].copy()
y_before = df['Target'].copy()
X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X_before, y_before, test_size=0.25, random_state=42)

model_before = LinearRegression()
model_before.fit(X_train_before, y_train_before)

y_pred_before = model_before.predict(X_test_before)
r2_before = r2_score(y_test_before, y_pred_before)





print(f"R^2 Score before Feature Engineering: {r2_before:.4f}")

R^2 Score before Feature Engineering: 0.8922


* _Note: The R^2 score for the linear regression model trained on the dataset without any further feature engineering is approximately 0.8877. This indicates that the model is able to explain about 88.77% of the variance in the test dataset._

This is a strong baseline, but it is not perfect because the model has not yet been given engineered representations of the categorical and temporal information, and some feature redundancy is still present.


### **TASK 4: One-Hot Encoding**

Convert the categorical feature (`Categorical`) to a numerical representation using one-hot encoding.
Display the resulting dataset by printing the first few rows of the dataframe.

### **Why encoding is necessary**

Linear regression requires numeric inputs. One-hot encoding converts each category into a binary indicator column so the model can learn a separate effect for each category level.

Use `drop_first=True` to avoid redundant columns and reduce multicollinearity among the dummy variables.


In [10]:
# Write your code here

# One-hot encoding for 'Categorical_1' using pandas `get_dummies()` method which will
# convert categorical columns into multiple columns of binary values (0 or 1) representing
# the presence of each possible category value.
df_encoded = pd.get_dummies(df, columns=['Categorical'], drop_first=True)

### **TASK 5: Scaling**

Scale and center the numerical features using Z-score normalization (`StandardScaler`).

### **Why scale features?**

Scaling places numeric columns on a comparable scale (mean near 0, standard deviation near 1), which is especially helpful when features have very different magnitudes.

Even when linear regression can still fit without scaling, normalization improves consistency and is a good habit for many ML pipelines.

Your resulting dataset should have the numerical columns (`x1`, `x2`, `x3`, and `x4`) scaled and centered.


In [ ]:
from sklearn.preprocessing import StandardScaler


# Write your code here

scale_columns = ['x1','x2','x3','x4']
scaler = StandardScaler()
df_encoded_scaled = df_encoded.copy()
df_encoded_scaled[scale_columns] = scaler.fit_transform(df_encoded_scaled[scale_columns])



#### **Drop Highly Correlated Features**

Highly correlated features can introduce **multicollinearity**, which makes linear model coefficients less stable and harder to interpret.

Here, `x4` was generated from `x1`, so it carries largely overlapping information. We drop `x4` before later modeling steps to reduce redundancy.


In [12]:
df_encoded_dropped = df_encoded.drop(columns=['x4'])

### **TASK 6: Extracting Temporal Data**

Extract date-related features (like year, month, day) from the `Temporal` column.

### **Why extract datetime components?**

Most machine learning models cannot directly use raw datetime values in a meaningful way. Breaking a timestamp into components helps expose patterns such as:

- seasonality or month effects
- day-of-month trends
- changes over time

In this dataset, datetime extraction is mainly practice for a common real-world feature engineering workflow.


In [13]:
# Write your code here
df_encoded_dropped['Year'] = df_encoded_dropped['Temporal'].dt.year
df_encoded_dropped['Month'] = df_encoded_dropped['Temporal'].dt.month
df_encoded_dropped['Day'] = df_encoded_dropped['Temporal'].dt.day


### **TASK 7: Dimensionality Reduction**

Identify and drop one or more highly correlated columns to reduce dimensionality. Showcase the correlation before and after this operation using a heatmap.

### **Goal of this step**

This is a simple form of feature selection: remove variables that provide nearly duplicate information. The heatmaps help confirm that the remaining numeric features are less correlated after the drop.


In [ ]:
# Write your code here

import seaborn as sns
import matplotlib.pyplot as plt


# Compute correlation matrix for the df_encoded DataFrame (before dropping)
correlation_before = ...


# Compute correlation matrix for the df_encoded_dropped DataFrame (after dropping)
correlation_after = ...

In [ ]:
# Visualize the correlation matrices
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.heatmap(correlation_before, annot=True, cmap='coolwarm', vmin=-1, vmax=1, cbar=False)
plt.title("Correlation Before Dropping 'x4'")

plt.subplot(1, 2, 2)
sns.heatmap(correlation_after, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation After Dropping 'x4'")

plt.tight_layout()
plt.show()

#correlation_before, correlation_after

### **Interpretation of Results**

**Correlation before dropping `x4`:**
- `x1` and `x4` have a very high correlation (approximately `0.99`), which indicates strong redundancy.
- The other numerical feature pairs show much lower correlations.

**Correlation after dropping `x4`:**
- Removing `x4` reduces multicollinearity in the numeric feature set.
- The remaining features (`x1`, `x2`, and `x3`) retain more distinct information, which can improve model stability and interpretability.


### **OPTIONAL TASK: Linear Regression Model After Feature Engineering**

Apply a linear regression model after the feature engineering tasks are done and compare its `R^2` score with that of the initial model.

### **What this comparison shows**

This final step evaluates whether the engineered feature representations (encoding, scaling, temporal extraction, and removing redundancy) help the model explain more variance in the target.


In [ ]:
# Compare R^2 score before and after Feature Engineering

plt.bar(['Before Feature Engineering', 'After Feature Engineering'], [r2_before, r2_after], color=['blue', 'green'])
plt.ylabel('R^2 Score')
plt.title('Comparison of R^2 Scores')
plt.show()

## **Summary and Key Takeaways**

### **What we practiced**

1. **EDA for feature planning**
- Identified data types and feature characteristics before modeling

2. **Baseline modeling**
- Trained a linear regression model to create a performance reference point

3. **Categorical encoding**
- Converted category labels into model-ready binary columns using one-hot encoding

4. **Feature scaling**
- Standardized numerical variables with Z-score normalization

5. **Datetime feature extraction**
- Derived year/month/day from a timestamp column for modeling use

6. **Reducing multicollinearity**
- Removed a highly correlated feature and validated the change with correlation heatmaps

### **Best Practices**

- Build a baseline model before feature engineering so improvements are measurable
- Match transformations to feature type (numeric, categorical, datetime)
- Watch for correlated features that can destabilize linear models
- Interpret feature engineering choices in terms of model performance and explainability
